In [ ]:
# https://docs.osmcode.org/pyosmium/latest/user_manual/04-Working-with-Filters/
import osmium
from osmium import filter, osm
import math

data= "../../denmark-260208.osm.pbf"
fp = osmium.FileProcessor(data) \
    .with_filter(filter.EntityFilter(osm.WAY)) \
    .with_filter(filter.KeyFilter("highway")
)

In [ ]:
def pythagoras(node1: osm.NodeRef, node2: osm.NodeRef):
    a = abs(node1.lon-node2.lon)
    b = abs(node1.lat-node2.lat)
    return math.hypot(a, b)

nodes: list[osm.NodeRef] = []
edges: list[tuple[osm.NodeRef, osm.NodeRef]] = [] # [(from, to)]
dists: list[float] = []

for way in fp:
    if not isinstance(way, osm.Way):
        raise AssertionError("bruh")
    
    valid_notes = [n for n in way.nodes if n.location.valid()]
    nodes.extend(valid_notes)

    # As long as we use a twoway network we dont need to check 'oneway' tag
    for n1, n2 in zip(valid_notes[:-1], valid_notes[1:]):
        edges.append((n1, n2))
        dists.append(pythagoras(n1, n2))

# Get node list index of all edges. (kinda stupid)
node_id_to_index = {n.ref: i for i, n in enumerate(nodes)}
edge_from_idx = [node_id_to_index[e[0].ref] for e in edges]
edge_to_idx   = [node_id_to_index[e[1].ref] for e in edges]

In [ ]:
import pandana as pdna
import pandas as pd

network = pdna.Network(
    pd.Series([n.lon for n in nodes], name="x"),
    pd.Series([n.lat for n in nodes], name="y"),
    pd.Series(edge_from_idx, name="from"),  # from node
    pd.Series(edge_to_idx, name="to"),  # to node
    pd.DataFrame({"dist": dists})  # impedance
)

In [ ]:
network.precompute(3000)